In [1]:
%pip install mediapipe==0.10.14 pyttsx3 requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import cv2, mediapipe, pyttsx3
print(cv2.__version__, mediapipe.__version__)

5.0.0 0.10.14


In [3]:
import getpass
TELEGRAM_TOKEN    = getpass.getpass("Bot token: ").strip()
TELEGRAM_CHAT_IDS = ["8849599822"]

In [4]:
import cv2, time, threading, collections, os, datetime
import mediapipe as mp
import pyttsx3, requests
from math import dist

DRIVER_NAME     = "Ramuji"
TRUCK_NO        = "UP32 XX 5276"
ALERT_SECONDS   = 3.0
PRE_SECONDS     = 15
POST_SECONDS    = 15
MAX_EVENT_SEC   = 120
NOTIFY_COOLDOWN = 120

# SMS is not used (leave empty)
TWILIO_SID   = ""
TWILIO_TOKEN = ""
TWILIO_FROM  = ""
OWNER_PHONE  = ""

LEFT  = [362, 385, 387, 263, 373, 380]
RIGHT = [33, 160, 158, 133, 153, 144]

face = mp.solutions.face_mesh.FaceMesh(
    max_num_faces=1, refine_landmarks=True,
    min_detection_confidence=0.5, min_tracking_confidence=0.5)

def get_points(lm, idxs, w, h):
    return [(lm[i].x * w, lm[i].y * h) for i in idxs]

def ear(p):
    return (dist(p[1], p[5]) + dist(p[2], p[4])) / (2 * dist(p[0], p[3]))

speaking = False
def speak():
    global speaking
    speaking = True
    try:
        try:
            import comtypes
            comtypes.CoInitialize()
        except Exception:
            pass
        eng = pyttsx3.init()
        eng.setProperty("rate", 150)
        for _ in range(2):
            eng.say(f"{DRIVER_NAME}, please take a rest.")
        eng.runAndWait()
        eng.stop()
    except Exception as e:
        print("Voice error:", e)
    speaking = False

def tg_ready():
    return (bool(TELEGRAM_TOKEN)
            and not TELEGRAM_TOKEN.startswith("PASTE")
            and all(str(c).lstrip("-").isdigit() for c in TELEGRAM_CHAT_IDS))

def send_telegram(text):
    if not tg_ready():
        print("[Telegram skipped: token or chat ID not set]")
        return
    for cid in TELEGRAM_CHAT_IDS:
        try:
            r = requests.post(f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage",
                              data={"chat_id": cid, "text": text}, timeout=10)
            print("Telegram text:", "sent" if r.ok else r.text)
        except Exception as e:
            print("Telegram error:", e)

def send_sms(text):
    if not TWILIO_SID:
        print("[SMS skipped: Twilio not set]")
        return
    try:
        from twilio.rest import Client
        Client(TWILIO_SID, TWILIO_TOKEN).messages.create(
            body=text, from_=TWILIO_FROM, to=OWNER_PHONE)
    except Exception as e:
        print("SMS error:", e)

def send_video(path, caption):
    if not tg_ready():
        return
    for cid in TELEGRAM_CHAT_IDS:
        try:
            with open(path, "rb") as f:
                r = requests.post(f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendVideo",
                                  data={"chat_id": cid, "caption": caption},
                                  files={"video": f}, timeout=120)
            print("Telegram video:", "sent" if r.ok else r.text)
        except Exception as e:
            print("Video send error:", e)

def notify_all(text):
    send_sms(text)
    send_telegram(text)

def save_clip(frames):
    if len(frames) < 2:
        return
    span = frames[-1][0] - frames[0][0]
    if span <= 0:
        return
    fps = max(5, min(30, len(frames) / span))
    os.makedirs("events", exist_ok=True)
    name = datetime.datetime.now().strftime("events/%Y%m%d_%H%M%S_") + DRIVER_NAME + ".mp4"
    first = cv2.imdecode(frames[0][1], cv2.IMREAD_COLOR)
    h, w = first.shape[:2]
    out = cv2.VideoWriter(name, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    for _, j in frames:
        out.write(cv2.imdecode(j, cv2.IMREAD_COLOR))
    out.release()
    print("Clip saved:", name)
    send_video(name, f"Drowsiness clip - {DRIVER_NAME}, truck {TRUCK_NO}")

cap = cv2.VideoCapture(0)
CALIB_TIME = 3
start = time.time()
calib, THRESH = [], 0.21
closed_since = None
last_closed_ts = 0.0
last_notify = 0.0
buf = collections.deque()
recording, event_frames, event_start = False, [], 0.0

while True:
    ok, frame = cap.read()
    if not ok:
        break
    frame = cv2.flip(frame, 1)
    now = time.time()
    _, jpg = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, 70])
    buf.append((now, jpg))
    while buf and now - buf[0][0] > PRE_SECONDS + ALERT_SECONDS + 2:
        buf.popleft()
    if recording:
        event_frames.append((now, jpg))

    h, w = frame.shape[:2]
    res = face.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    status, color = "No face", (200, 200, 200)
    eyes_closed, closed_for = False, 0.0

    if res.multi_face_landmarks:
        lm = res.multi_face_landmarks[0].landmark
        e = (ear(get_points(lm, LEFT, w, h)) +
             ear(get_points(lm, RIGHT, w, h))) / 2
        if now - start < CALIB_TIME:
            calib.append(e)
            status, color = "Calibrating...Keep Eye Open", (0, 255, 255)
        else:
            if calib:
                THRESH = 0.75 * (sum(calib) / len(calib))
                calib = []
            eyes_closed = e < THRESH
            status, color = (("Eyes closed", (0, 165, 255)) if eyes_closed
                             else ("Awake", (0, 255, 0)))
        cv2.putText(frame, f"EAR {e:.2f} / Thresh {THRESH:.2f}", (10, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    if eyes_closed:
        last_closed_ts = now
        if closed_since is None:
            closed_since = now
        closed_for = now - closed_since
    else:
        closed_since = None

    if closed_for >= ALERT_SECONDS:
        status, color = "WAKE UP!", (0, 0, 255)

        if not speaking:
            threading.Thread(target=speak, daemon=True).start()

        if now - last_notify > NOTIFY_COOLDOWN:
            last_notify = now
            msg = (f"ALERT: Driver {DRIVER_NAME} (truck {TRUCK_NO}) looks drowsy "
                   f"at {datetime.datetime.now():%H:%M}. "
                   f"Please be alert and call the driver.")
            threading.Thread(target=notify_all, args=(msg,), daemon=True).start()

        if not recording:
            recording, event_start = True, now
            event_frames = [(t, j) for t, j in buf
                            if t >= closed_since - PRE_SECONDS]

    if recording and (now - last_closed_ts >= POST_SECONDS
                      or now - event_start >= MAX_EVENT_SEC):
        threading.Thread(target=save_clip, args=(event_frames,), daemon=True).start()
        recording, event_frames = False, []

    if recording:
        cv2.circle(frame, (w - 30, 30), 10, (0, 0, 255), -1)
        cv2.putText(frame, "REC", (w - 90, 38),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    cv2.putText(frame, status, (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 3)
    cv2.imshow("Drowsiness Alert (Esc = exit)", frame)
    if cv2.waitKey(1) == 27:
        break

if recording:
    save_clip(event_frames)
cap.release()
cv2.destroyAllWindows()

c:\Users\ROHIT MAURYA\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


[SMS skipped: Twilio not set]
Telegram text: sent
Clip saved: events/20261104_121909_Ramuji.mp4
Telegram video: sent
